# Sentiment Analysis: Visualisations

This notebook produces all result plots for the Cornell Movie Review sentiment analysis project. All data is coded directly from experiment results, no external files are required.

**Plots generated:**
1. Feature Ablation Study (grouped bar chart)
2. Full Model Comparison (grouped bar chart)
3. BERT Training Loss Convergence (line plot)
4. Confusion Matrix Grid (2 × 3 subplot grid)

All plots are saved to the `assets/` folder.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

# global style
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 100, "savefig.bbox": "tight"})

# resolve assets/ directory regardless of CWD
if os.path.isdir("notebooks"):        # running from project root
    ASSETS_DIR = "assets"
else:                                  # running from notebooks/ folder
    ASSETS_DIR = "../assets"
os.makedirs(ASSETS_DIR, exist_ok=True)
print(f"Assets will be saved to: {os.path.abspath(ASSETS_DIR)}")

In [ ]:
feature_ablation_results = {
    "Trigrams + Freq Norm": {"Accuracy": 0.57, "Precision": 0.54, "Recall": 0.92, "F1": 0.68},
    "Bigrams + TF-IDF":     {"Accuracy": 0.70, "Precision": 0.69, "Recall": 0.70, "F1": 0.70},
    "Unigrams + TF-IDF":    {"Accuracy": 0.84, "Precision": 0.80, "Recall": 0.90, "F1": 0.85},
}

model_comparison_results = {
    "Naïve Bayes\n(scratch)":   {"Accuracy": 0.84, "Precision": 0.80, "Recall": 0.90, "F1": 0.85},
    "MultinomialNB\n(sklearn)": {"Accuracy": 0.84, "Precision": 0.81, "Recall": 0.89, "F1": 0.85},
    "Logistic\nRegression":     {"Accuracy": 0.84, "Precision": 0.83, "Recall": 0.87, "F1": 0.85},
    "SVM\n(LinearSVC)":         {"Accuracy": 0.85, "Precision": 0.84, "Recall": 0.86, "F1": 0.85},
    "BERT\nCased":              {"Accuracy": 0.89, "Precision": 0.89, "Recall": 0.89, "F1": 0.89},
    "BERT\nUncased":            {"Accuracy": 0.91, "Precision": 0.91, "Recall": 0.91, "F1": 0.91},
}

bert_training_loss = {
    "Uncased": {250: 0.5719, 500: 0.5585, 750: 0.5234, 1000: 0.4311,
                1250: 0.3827, 1500: 0.2036, 1750: 0.1734},
    "Cased":   {250: 0.5953, 500: 0.5753, 750: 0.5344, 1000: 0.4647,
                1250: 0.3672, 1500: 0.1817, 1750: 0.1477},
}

print("Data loaded.")
print(f"  Feature sets : {list(feature_ablation_results.keys())}")
print(f"  Models       : {list(model_comparison_results.keys())}")

In [ ]:
# Plot 1: Feature Ablation grouped bar chart
metrics      = ["Accuracy", "Precision", "Recall", "F1"]
feature_sets = list(feature_ablation_results.keys())
n_groups     = len(feature_sets)
n_metrics    = len(metrics)
x            = np.arange(n_groups)
width        = 0.18
palette      = sns.color_palette("muted", n_metrics)

fig, ax = plt.subplots(figsize=(11, 6))

for i, metric in enumerate(metrics):
    vals = [feature_ablation_results[fs][metric] for fs in feature_sets]
    bars = ax.bar(
        x + (i - (n_metrics - 1) / 2) * width, vals, width,
        label=metric, color=palette[i], edgecolor="white"
    )
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.007,
            f"{v:.2f}", ha="center", va="bottom", fontsize=8.5
        )

ax.set_xticks(x)
ax.set_xticklabels(feature_sets, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Feature Ablation Study \u2014 Na\u00efve Bayes (Development Set)",
             fontsize=13, pad=12)
ax.legend(title="Metric", fontsize=10)
ax.yaxis.grid(True, alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
save_path = os.path.join(ASSETS_DIR, "feature_ablation.png")
plt.savefig(save_path, dpi=150)
print(f"Saved: {save_path}")
plt.show()

In [ ]:
# Plot 2: Full Model Comparison grouped bar chart
models           = list(model_comparison_results.keys())
n_models         = len(models)
x                = np.arange(n_models)
palette          = sns.color_palette("muted", n_metrics)
BERT_UNCASED_IDX = models.index("BERT\nUncased")
HIGHLIGHT_COLOR  = "#FFC107"   # amber / gold for BERT Uncased

fig, ax = plt.subplots(figsize=(14, 6))
all_bars = []

for i, metric in enumerate(metrics):
    vals = [model_comparison_results[m][metric] for m in models]
    bar_colors = [
        HIGHLIGHT_COLOR if j == BERT_UNCASED_IDX else palette[i]
        for j in range(n_models)
    ]
    bars = ax.bar(
        x + (i - (n_metrics - 1) / 2) * width, vals, width,
        color=bar_colors, edgecolor="white", label=metric
    )
    all_bars.append(bars)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.004,
            f"{v:.2f}", ha="center", va="bottom", fontsize=7
        )

# Classical ceiling dashed line
ax.axhline(0.85, linestyle="--", color="dimgray", linewidth=1.5,
           label="Classical ceiling (F1 = 0.85)")

# Star annotation above BERT Uncased group
ax.annotate(
    "\u2605 Best", xy=(BERT_UNCASED_IDX, 0.935),
    ha="center", fontsize=10, fontweight="bold", color="darkorange"
)

# Legend with highlight patch
highlight_patch = mpatches.Patch(color=HIGHLIGHT_COLOR, label="BERT Uncased (highlighted)")
handles, labels_leg = ax.get_legend_handles_labels()
handles.append(highlight_patch)
ax.legend(handles=handles, fontsize=9, loc="lower right")

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Model Comparison \u2014 All Models on Test Set", fontsize=13, pad=12)
ax.yaxis.grid(True, alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
save_path = os.path.join(ASSETS_DIR, "model_comparison.png")
plt.savefig(save_path, dpi=150)
print(f"Saved: {save_path}")
plt.show()

In [ ]:
# Plot 3: BERT Training Loss Curve
fig, ax = plt.subplots(figsize=(9, 5))

styles = {
    "Uncased": {"color": "#1976D2", "marker": "o", "linestyle": "-"},
    "Cased":   {"color": "#E64A19", "marker": "s", "linestyle": "--"},
}

for variant, losses in bert_training_loss.items():
    steps = sorted(losses.keys())
    vals  = [losses[s] for s in steps]
    ax.plot(steps, vals, label=f"BERT {variant}",
            linewidth=2.5, markersize=8, **styles[variant])

ax.set_xlabel("Training Step", fontsize=12)
ax.set_ylabel("Training Loss", fontsize=12)
ax.set_title("BERT Training Loss Convergence", fontsize=13, pad=12)
ax.set_xticks(sorted(bert_training_loss["Uncased"].keys()))
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)

plt.tight_layout()
save_path = os.path.join(ASSETS_DIR, "bert_training_loss.png")
plt.savefig(save_path, dpi=150)
print(f"Saved: {save_path}")
plt.show()

In [ ]:
# Plot 4: Confusion Matrix Grid (2 × 3)
confusion_matrices = {
    "NB (scratch)":  np.array([[318,  88], [ 39, 355]]),
    "MultinomialNB": np.array([[326,  80], [ 45, 349]]),
    "LR (tuned)":    np.array([[330,  77], [ 37, 356]]),
    "SVM (tuned)":   np.array([[332,  75], [ 34, 359]]),
    "BERT Cased":    np.array([[357,  50], [ 38, 355]]),
    "BERT Uncased":  np.array([[370,  37], [ 24, 369]]),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for ax, (name, cm) in zip(axes.flatten(), confusion_matrices.items()):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Negative", "Positive"]
    )
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=12, pad=8)

plt.suptitle("Confusion Matrices \u2014 All Models on Test Set",
             fontsize=14, y=1.01)
plt.tight_layout()

save_path = os.path.join(ASSETS_DIR, "confusion_matrix_grid.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved: {save_path}")
plt.show()

## Key Visual Takeaways

### 1. Feature Ablation (Plot 1)
- **Representation quality dominates**: switching from trigrams (F1 = 0.68) to unigrams (F1 = 0.85) improves F1 by **17 points**, far larger than any classifier-level gain.
- Trigram frequency normalisation achieves very high recall (0.92) but low precision (0.54), biasing heavily toward positive predictions.
- TF-IDF weighting consistently improves the precision–recall balance.

### 2. Model Comparison (Plot 2)
- All four classical models plateau at **F1 = 0.85**, confirming a bag-of-words ceiling imposed by discarding word order.
- BERT Cased (+4 pp, F1 = 0.89) and **BERT Uncased (+6 pp, F1 = 0.91)** break through this ceiling by encoding full contextual embeddings.
- BERT Uncased outperforms Cased: case sensitivity appears to add vocabulary noise without proportional discriminative benefit for sentiment tasks.

### 3. BERT Loss Curve (Plot 3)
- Both models converge smoothly over 1,750 steps (~72 % loss reduction).
- Uncased reaches a slightly lower final loss, consistent with its superior test performance.
- No loss spikes are visible, indicating stable fine-tuning.

### 4. Confusion Matrices (Plot 4)
- Classical models show broadly balanced false-positive / false-negative errors.
- BERT Uncased dramatically reduces both error types compared to NB scratch (FP: 37 vs 88, FN: 24 vs 39).
- Across all models, false negatives (missed positives) are fewer than false positives.